# 01 Data exploration
Run this after `scripts/download_data.py` (raw) and `scripts/preprocess_data.py` (processed). Outputs are not saved in this file: run the cells to see your own results, then write the findings into the README.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, plotly.express as px
from src import config, data_loader, preprocessing
paths = config.get_paths()
raw = pd.read_csv(paths.data_raw / 'city_day.csv')
raw.shape, raw.dtypes

## Coverage and missingness

In [ ]:
print('Date range:', raw['Date'].min(), '->', raw['Date'].max())
print('Cities:', raw['City'].nunique())
print((raw.isna().mean()*100).round(1).sort_values(ascending=False))
print('Duplicate City+Date rows:', raw.duplicated(['City','Date']).sum())

In [ ]:
df, report = preprocessing.clean_air_quality(raw)
report

In [ ]:
preprocessing.detect_gaps(df).sort_values('missing_days')

## Distributions, trends, seasonality

In [ ]:
px.histogram(df, x='pm25', nbins=100, title='PM2.5 distribution (µg/m³)')

In [ ]:
d = df[df.city.isin(df.groupby('city').pm25.count().nlargest(5).index)]
px.line(d, x='date', y='pm25', color='city', title='PM2.5 over time')

In [ ]:
m = df.assign(month=df.date.dt.month).groupby(['city','month']).pm25.mean().reset_index()
px.line(m, x='month', y='pm25', color='city', title='Mean PM2.5 by month')

## Outliers
Extreme values are inspected, not deleted. Check whether they cluster in time (plausible event) or look like isolated errors.

In [ ]:
q99 = df.pm25.quantile(0.99)
df[df.pm25 > q99].groupby('city').size().sort_values(ascending=False)

## Weather correlation (needs `scripts/download_weather.py`)

In [ ]:
wx = data_loader.load_weather_many(sorted(df.city.unique()))
if wx:
    j = pd.concat([df[df.city==c].merge(w, on='date') for c, w in wx.items()])
    display(j[['pm25','pm10','temp','rh','wind','precip','pressure']].corr().round(2))
else:
    print('No weather cached yet')

## Suitability for forecasting
Observations are **daily**. This supports a 7-day daily forecast. It does not support a validated hourly forecast, so none is built.